In [2]:
# today we look at parallel function calling
import os
import time
from langchain_core.tools import tool
from dotenv import load_dotenv, find_dotenv

# ddefining three distinct tools

@tool
def get_material_cost(material_type: str) -> str:
    """Gets the current market cost for a specific construction material."""
    time.sleep(2)  # simulating network/ database latency
    costs = {"cement": "35,000 UGX per bag", "bricks": "250 UGX per brick"}
    return costs.get(material_type.lower(), "Cost unknown")

@tool
def get_labor_rate(role: str) -> str:
    """Gets the standard daily wage for a specific construction role."""
    time.sleep(2) # simulating network /database latency
    rates = { "mason": "40,000 UGX/day", "porter": "15,000 UGX/day"}
    return rates.get(role.lower(), "Rate unknown")

@tool
def get_weather_forecast(location: str) -> str:
    """Gets weather forecast for a specific location to determine if work can proceed."""
    time.sleep(2) # simulating network/database latency
    return f"Forecast for {location}: 80% chance of heavy rain, site work not recommended"

tools = [get_material_cost, get_labor_rate, get_weather_forecast]

In [3]:
# loading api keys

load_dotenv(find_dotenv())

True

In [4]:
# binding tools to the llm

from langchain_groq import ChatGroq

# initiating an llm ( must be one that supports parallel tool calling)
llm = ChatGroq(
    model= 'llama-3.3-70B-versatile',
    api_key= os.getenv('GROQ_API_KEY'),
    temperature= 0
)

#binding the tools with the llm
llmWithTools = llm.bind_tools(tools= tools)

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Triggering parallel Calls
# we will send a single prompt asking for three different data points to force the llm to use all three tools.

# the triple prompt
query = (
    "I'm plannin to build in Lugazi tomorrow."
    "I need to know the cost of cement, the daily rate for a mason, "
    "and the weather forecast for Lugazi to see if we can pour the foundation"
)

print(f"Sending Query: '{query}'\n")

# Call the llm
start_time = time.time()
ai_msg = llmWithTools.invoke(query)
endTime = time.time()

print(f"LLM decision time: {endTime - start_time:.2f} seconds")
# Checking if the llm decided to call multiple tools
if ai_msg.tool_calls:
    print(f"\nThe agent decided to trigger {len(ai_msg.tool_calls)} tools simultaneously:")
    for tool_call in ai_msg.tool_calls:
        print(f" - Tool: {tool_call['name']} | Args: {tool_call['args']}")
else:
    print("The agent did not call any tools.")

Sending Query: 'I'm plannin to build in Lugazi tomorrow.I need to know the cost of cement, the daily rate for a mason, and the weather forecast for Lugazi to see if we can pour the foundation'

LLM decision time: 0.45 seconds

The agent decided to trigger 3 tools simultaneously:
 - Tool: get_material_cost | Args: {'material_type': 'cement'}
 - Tool: get_labor_rate | Args: {'role': 'mason'}
 - Tool: get_weather_forecast | Args: {'location': 'Lugazi'}


In [8]:
# executing the tools concurrently

import concurrent.futures

# mapping the tool names back to the actual python functions
toolMap = {tool.name: tool for tool in tools}

def execute_tool(tool_call):
    '''function to execute a single toolc call'''
    selectedTool = toolMap[tool_call['name']]
    # Execute the tool witht the provided arguments
    result = selectedTool.invoke(tool_call['args'])
    return f"Result from {tool_call['name']}: {result}"

print("\n--- Executing Tools in parallel ---")
execStartTime = time.time()

# running the tools concurrently 
results = []
if ai_msg.tool_calls:
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # submit all tool calls to the executor
        futureToTool ={
            executor.submit(execute_tool, tool_call): tool_call
            for tool_call in ai_msg.tool_calls
        }

        # Gather results as they complete
        for future in concurrent.futures.as_completed(futureToTool):
            results.append(future.result())

execEndTime = time.time()
# print the final results and the time saved
for res in results:
    print(res)

print(f"\nTotal Execution time for Tools: {execEndTime - execStartTime: .2f} seconds")
print("Note: if run sequentially, this would have taken ~6 seconds!")



--- Executing Tools in parallel ---
Result from get_material_cost: 35,000 UGX per bag
Result from get_labor_rate: 40,000 UGX/day
Result from get_weather_forecast: Forecast for Lugazi: 80% chance of heavy rain, site work not recommended

Total Execution time for Tools:  2.01 seconds
Note: if run sequentially, this would have taken ~6 seconds!
